# Sequential Counter Encoding

## 1. Giới thiệu
Cardinality Constraint (Ràng buộc số lượng) là ràng buộc kiểu "nhiều nhất, ít nhất, đúng k biến đúng".  Cách cơ bản là cấm tất cả các tổ hợp k+1 biến đúng nhưng sẽ bùng nổ tập mệnh đề khiến cho bài toán gần như không thể tìm ra nghiệm trong thời gian được giới hạn. 

Ý tưởng chung để encode một cách hiệu quả là đếm rồi so sánh. Có hai stage cơ bản:
1. Counter stage
2. Comparator stage

### 1.1 Counter stage 
Counter stage là phần đếm xem có bao nhiêu biến là true (tương tự như một mảng tổng tiền tố)
Công thức như sau: $$S_i = \sum_{j=1}^{i}x_j, \quad \forall 1 \le i < n  $$

### 1.2 Comparator stage
Comparator stage là phần so sánh tổng vừa đếm được với k. Nếu constraint là: $ \leq k(x_1, \dots, x_n) $ thì comparator kiểm tra: $$S_i \leq k, \quad \forall 1 \leq i < n$$


## 2. Các định nghĩa cơ bản
Có 5 loại constraint được các tác giả định nghĩa:
1. $ \le k(\phi_{1}, \dots, \phi_{n})$: nhiều nhất $k$ công thức đúng. Ràng buộc này luông đúng với $k \ge n$ và luôn sai với $k<0$
2. $ \ge k(\phi_{1}, \dots, \phi_{n})$: ít nhất $k$ công thức đúng. Ràng buộc này luôn đúng với $k \le 0$ và luôn sai với $k>n$
3. $ = k(\phi_{1}, \dots, \phi_{n})$: đúng $k$ công thức đúng.
4. $ \le k(\phi_{1}, \dots, \phi_{n})$ tương đương $\ge (n-k)(\neg x_1, \dots, \neg x_n)$
5. $ \neg  \le k(\phi_{1}, \dots, \phi_{n})$ tương đương $\ge (k+1)(x_1,\dots,x_n)$

Có hai điều quan trọng là các tác giả đã chứng minh rằng chỉ cần tập trung vào constraint 1 vì: $$\ge k(x_{1}, \dots,  x_{n})$$ có thể đổi thành $$\le (n-k)(\neg x_{1}, \dots, \neg x_{n})$$ và constraint 3 là sự kết hợp của constraint 1 và constraint 2. 





## 3. Mã hóa bằng Sequential Counter

### 3.1. Giải thích sơ bộ

Để giải bài toán: $$\le k(x_1,\dots, x_n)$$ nghĩa là trong $n$ biến, không được có quá $k$ biến bằng `true`, tác giả xây một sequential counter đếm đi lần lượt từ trái sang phải.


![image.png](./images/SequentialCounter/image.png)

#### 3.1.1. Hình bên trái
Ở hình bên trái, đếm tuần tự: $x_1 \rightarrow x_2 \rightarrow \dots \rightarrow x_n$. Mỗi trạng thái $s_i$ gồm $k$ bit: $$s_{i,1}, s_{i,2},\dots,s_{i,k}$$ Trong đó: $s_{i,j}=1$ có ý nghĩa là trong $x_1,\dots,x_n$ có ít nhất 1 biến `true`. Ví dụ với $k=3$: $s_i=(1,1,0)$ thì điều này có nghĩa là đã có ít nhất 2 biến `true` nhưng chưa tới 3 nên $s_i = 0$

#### 3.1.2. Hình bên phải
Ở hình bên phải, mạch nhận vào:
- biến mới $x_i$
- trạng thái cũ: $s_{i-1,1},s_{i-1,2},\dots,s_{i-1,k}$

Từ đầu vào, mạch tạo ra:
- trạng thái mới: $s_{i,1},s_{i,2},\dots,s_{i,k}$
- cùng bit tràn: $v_i$

Các cổng trong hình có ý nghĩa như sau:
- $\ge 1$ nghĩa là `or`
- `&` nghĩa là `and`

Trong khối này thực hiện 3 công thức sau:
1. $s_{i,1} \iff x_i \lor s_{i-1,i}$:

| Chiều thuận ($\rightarrow$) | Chiều nghịch ($\leftarrow$) |
| :--- | :--- |
| **Mệnh đề:** $s_{i,1} \rightarrow (x_i \lor s_{i-1,1})$ | **Mệnh đề:** $(x_i \lor s_{i-1,1}) \rightarrow s_{i,1}$ |
| **Ý nghĩa:** Nếu ghi nhận có ít nhất một giá trị `true` tại bước $i$ thì bắt buộc phải có "nguồn gốc": <br> 1. Hoặc biến hiện tại $x_i$ là `true`. <br> 2. Hoặc trước đó ($s_{i-1,1}$) đã có `true`. | **Ý nghĩa:** Nếu thực tế tồn tại một giá trị `true` thì biến trạng thái $s_{i,1}$ bắt buộc phải bật lên để phản ánh đúng thực tế đó: <br> 1. Nếu $x_i = 1$ $\rightarrow$ $s_{i,1} = 1$. <br> 2. Nếu $s_{i-1,1} = 1$ $\rightarrow$ $s_{i,1} = 1$. |
| **Mục đích:** Ngăn chặn việc biến trạng thái tự ý nhận giá trị `true` vô căn cứ (tránh "true nhầm"). | **Mục đích:** Đảm bảo tính lan truyền và kế thừa giá trị logic xuyên suốt chuỗi biến. |


2. $s_{i,j} \iff s_{i-1,j} \lor (x_i \land s_{i-1,j-1}), \quad j>1$: 

| Chiều thuận ($\rightarrow$) | Chiều nghịch ($\leftarrow$) |
| :--- | :--- |
| **Mệnh đề:** $s_{i,j} \rightarrow s_{i-1,j} \lor (x_i \land s_{i-1,j-1})$ | **Mệnh đề:** $s_{i-1,j} \lor (x_i \land s_{i-1,j-1}) \rightarrow s_{i,j}$ |
| **Ý nghĩa:** Nếu tại bước $i$ ta có ít nhất $j$ biến `true` thì chỉ có hai khả năng đã xảy ra: <br> 1. **Kế thừa:** Ngay từ bước $i-1$ đã có đủ $j$ biến `true` ($s_{i-1,j}$). <br> 2. **Tích lũy mới:** Bước $i-1$ mới có $j-1$ biến `true` và biến hiện tại $x_i$ vừa vặn là biến `true` thứ $j$. | **Ý nghĩa:** Nếu một trong hai điều kiện sau thỏa mãn, trạng thái ít nhất $j$ biến `true` tại bước $i$ phải được xác nhận: <br> 1. Nếu trước đó đã có đủ $j$ biến `true` thì hiển nhiên bước này vẫn có ít nhất $j$ biến. <br> 2. Nếu đã có $j-1$ biến và biến mới $x_i$ cũng là `true` thì tổng cộng đã đạt mốc $j$. |
| **Mục đích:** Đảm bảo biến trạng thái $s_{i,j}$ không tự ý bật lên nếu không thỏa mãn các điều kiện đếm tích lũy. | **Mục đích:** Bắt buộc bộ giải SAT phải cập nhật trạng thái đếm ngay khi điều kiện về số lượng biến `true` được thỏa mãn. |

3. $v_i \iff x_i \land s_{i-1,k}$. 

| Chiều thuận ($\rightarrow$) | Chiều nghịch ($\leftarrow$) |
| :--- | :--- |
| **Mệnh đề:** $v_i \rightarrow x_i \land s_{i-1,k}$ | **Mệnh đề:** $(x_i \land s_{i-1,k}) \rightarrow v_i$ |
| **Ý nghĩa (Điều kiện cần):** <br> Nếu ghi nhận có sự vi phạm ($v_i$) xảy ra tại bước $i$ thì bắt buộc hai điều sau phải cùng xảy ra: <br> 1. Biến hiện tại $x_i$ phải là `true`. <br> 2. Trước đó đã đạt ngưỡng tối đa là $k$ biến `true` ($s_{i-1,k}$). | **Ý nghĩa (Điều kiện đủ):** <br> Nếu thực tế tại bước $i$ ta có $x_i = 1$ và trước đó đã tích lũy đủ $k$ biến `true` thì hệ thống bắt buộc phải kích hoạt biến vi phạm $v_i$. |
| **Mục đích:** Đảm bảo vi phạm chỉ được báo cáo nếu thực sự có biến thứ $k+1$ xuất hiện. | **Mục đích:** Ngăn chặn việc bộ giải SAT lờ đi sự vi phạm; ép giá trị $v_i$ lên `true` để hệ thống nhận diện lỗi. |



Tuy nhiên, vì đây là bài toán At most k nên chúng ta chỉ quan tâm đến việc ngăn chặn sự vi phạm, tức là không cho phép $v_i$ được bật lên. Do đó, điều kiện cuối cùng để đảm bảo ràng buộc được thỏa mãn là: $$\neg v_i, \quad \forall 1 \le i \le n$$
tức là không có bước nào được phép báo vi phạm. 

Ngoài ra, tác giả còn nhận xét rằng $s_{i,j}$ chỉ là biến phụ nhằm đưa ra tín hiệu "đã đạt ngưỡng" nên phải giữ chiều $A \rightarrow s_{i,j}$ để không bỏ sót trường hợp nào. Với chiều ngược lại, nếu $s_{i,j}$ tự bật lên mà không có nguồn gốc rõ ràng thì cũng không ảnh hưởng đến tính đúng đắn của giải pháp vì nó chỉ là tín hiệu phụ, không phải là điều kiện bắt buộc để đạt được ràng buộc.  

### 3.2. At least k

Mục tiêu của phần này là mã hóa ràng buộc:

$$
\sum_{t=1}^{n} x_t \ge k
$$

Biến phụ $s_{i,j}$ có nghĩa là trong prefix $x_1,\dots,x_i$ đã có **ít nhất $j$ biến đúng**.

#### 1. Trường hợp cơ sở

$$
(\neg x_1 \lor s_{1,1}) \quad \land \quad (\neg s_{1,1} \lor x_1)
$$

Tương đương với:

$$
s_{1,1} \leftrightarrow x_1
$$

Nghĩa là với prefix đầu tiên, có ít nhất một biến đúng khi và chỉ khi $x_1$ đúng.

#### 2. Cập nhật trạng thái $s_{i,1}$

Với mọi $2 \le i \le n$:

$$
\begin{aligned}
(\neg x_i \lor s_{i,1}) &\quad &&x_i \rightarrow s_{i,1} \\
(\neg s_{i-1,1} \lor s_{i,1}) &\quad &&s_{i-1,1} \rightarrow s_{i,1} \\
(\neg s_{i,1} \lor x_i \lor s_{i-1,1}) &\quad &&s_{i,1} \rightarrow (x_i \lor s_{i-1,1})
\end{aligned}
$$

Ba clause này nói rằng prefix hiện tại có ít nhất một biến đúng nếu $x_i$ đúng hoặc prefix trước đó đã có ít nhất một biến đúng.

#### 3. Cập nhật trạng thái tổng quát $s_{i,j}$

Với mọi $2 \le i \le n$ và $2 \le j \le \min(i-1,k)$:

$$
\begin{aligned}
(\neg s_{i-1,j} \lor s_{i,j})
&\quad &&s_{i-1,j} \rightarrow s_{i,j} \\
(\neg x_i \lor \neg s_{i-1,j-1} \lor s_{i,j})
&\quad &&x_i \land s_{i-1,j-1} \rightarrow s_{i,j}
\end{aligned}
$$

Hai clause trên tạo chiều thuận: nếu prefix trước đã đạt $j$, hoặc thêm $x_i$ vào prefix đã đạt $j-1$, thì prefix hiện tại đạt $j$.

Chiều ngược dùng hai clause:

$$
\begin{aligned}
(\neg s_{i,j} \lor s_{i-1,j} \lor x_i) \\
(\neg s_{i,j} \lor s_{i-1,j} \lor s_{i-1,j-1})
\end{aligned}
$$

Tương đương với:

$$
s_{i,j} \rightarrow \Bigl(s_{i-1,j} \lor (s_{i-1,j-1} \land x_i)\Bigr)
$$

Tức là nếu prefix hiện tại đã đạt $j$ biến đúng, trạng thái đó phải đến từ một nguồn hợp lệ.

#### 4. Đường chéo $s_{i,i}$

Với mọi $2 \le i \le k$:

$$
(\neg s_{i-1,i-1} \lor \neg x_i \lor s_{i,i})
$$

Tương đương với:

$$
s_{i-1,i-1} \land x_i \rightarrow s_{i,i}
$$

Chiều ngược:

$$
\begin{aligned}
(\neg s_{i,i} \lor s_{i-1,i-1}) \\
(\neg s_{i,i} \lor x_i)
\end{aligned}
$$

Tức là:

$$
s_{i,i} \rightarrow (s_{i-1,i-1} \land x_i)
$$

Nếu đã có ít nhất $i$ biến đúng trong $i$ biến đầu, thì toàn bộ $i$ biến đầu đều phải đúng.

#### 5. Điều kiện cuối

$$
s_{n,k}
$$

Điều kiện này ép toàn bộ dãy có ít nhất $k$ biến đúng.

#### 6. Miền biến phụ

Chỉ cần tạo các biến:

$$
s_{i,j}, \quad 1 \le j \le \min(i,k)
$$

Cách này tránh các trạng thái bất khả như $s_{1,2}$, $s_{2,3}$, ... vì với $i$ biến thì không thể có ít nhất $j$ biến đúng nếu $j>i$.

In [ ]:
class VarPool:
    """Small helper for allocating fresh CNF variable ids."""

    def __init__(self, top=0):
        self.top = top

    def new_var(self):
        self.top += 1
        return self.top


def ALK(clauses, variables, pool, k):
    """Encode At Least K using a sequential counter.

    Args:
        clauses: list[list[int]], mutated in-place.
        variables: CNF variable ids, e.g. [1, 2, 3]. Indices are 0-based.
        pool: VarPool, used to allocate fresh auxiliary variables.
        k: required lower bound.

    Returns:
        dict[(int, int), int]: mapping (i, j) -> auxiliary variable s_{i,j}.
        Here s[(i, j)] means variables[0..i] contain at least j + 1 true variables.
    """
    n = len(variables)

    if k <= 0:
        return {}

    if k > n:
        clauses.append([])  # UNSAT: cannot choose at least k variables from n variables.
        return {}

    

    s = {}


    # Create only valid states: s[(i, j)], 0 <= j < min(i + 1, k).
    for i in range(n):
        for j in range(min(i + 1, k)):
            s[(i, j)] = pool.new_var()

    x_0 = variables[0]
    clauses.append([-x_0, s[(0, 0)]])
    clauses.append([-s[(0, 0)], x_0])

    for i in range(1, n):
        x_i = variables[i]

        # s[(i, 0)] <-> (x_i or s[(i - 1, 0)])
        clauses.append([-x_i, s[(i, 0)]])
        clauses.append([-s[(i - 1, 0)], s[(i, 0)]])
        clauses.append([-s[(i, 0)], x_i, s[(i - 1, 0)]])

        # General update for s[(i, j)] where 0 < j < i.
        for j in range(1, min(i - 1, k - 1) + 1):
            clauses.append([-s[(i - 1, j)], s[(i, j)]])
            clauses.append([-x_i, -s[(i - 1, j - 1)], s[(i, j)]])
            clauses.append([-s[(i, j)], s[(i - 1, j)], x_i])
            clauses.append([-s[(i, j)], s[(i - 1, j)], s[(i - 1, j - 1)]])

        # Diagonal state: s[(i, i)] <-> (s[(i - 1, i - 1)] and x_i).
        if i < k:
            clauses.append([-s[(i - 1, i - 1)], -x_i, s[(i, i)]])
            clauses.append([-s[(i, i)], s[(i - 1, i - 1)]])
            clauses.append([-s[(i, i)], x_i])

    clauses.append([s[(n - 1, k - 1)]])
    return s

### 3.3. At most k

Mục tiêu của phần này là mã hóa ràng buộc:

$$
\sum_{t=1}^{n} x_t \le k
$$

Biến phụ $s_{i,j}$ có nghĩa là trong prefix $x_1,\dots,x_i$ đã có **ít nhất $j$ biến đúng**. Với AMK, ta chỉ cần tạo trạng thái cho các prefix trước phần tử cuối:

$$
1 \le i < n, \qquad 1 \le j \le \min(i,k)
$$

#### 1. Trường hợp cơ sở

$$
x_1 \rightarrow s_{1,1}
$$

CNF:

$$
(\neg x_1 \lor s_{1,1})
$$

Nếu $x_1$ đúng thì prefix đầu tiên đã có ít nhất một biến đúng.

#### 2. Lan truyền tầng $j=1$

Với mọi $1 < i < n$:

$$
\begin{aligned}
(\neg x_i \lor s_{i,1}) &\quad &&x_i \rightarrow s_{i,1} \\
(\neg s_{i-1,1} \lor s_{i,1}) &\quad &&s_{i-1,1} \rightarrow s_{i,1}
\end{aligned}
$$

Hai clause này đảm bảo nếu $x_i$ đúng, hoặc prefix trước đó đã có một biến đúng, thì $s_{i,1}$ phải bật lên.

#### 3. Lan truyền tổng quát

Với mọi $1 < i < n$ và $2 \le j \le \min(i-1,k)$:

$$
\begin{aligned}
(\neg s_{i-1,j} \lor s_{i,j})
&\quad &&s_{i-1,j} \rightarrow s_{i,j} \\
(\neg s_{i-1,j-1} \lor \neg x_i \lor s_{i,j})
&\quad &&s_{i-1,j-1} \land x_i \rightarrow s_{i,j}
\end{aligned}
$$

Clause thứ nhất giữ tính kế thừa: đã đạt $j$ biến đúng thì các prefix sau vẫn đạt $j$. Clause thứ hai cập nhật bộ đếm khi thêm một biến đúng mới.

#### 4. Chặn tràn

Với mọi $1 < i \le n$:

$$
x_i \rightarrow \neg s_{i-1,k}
$$

CNF:

$$
(\neg x_i \lor \neg s_{i-1,k})
$$

Ý nghĩa: nếu trước vị trí $i$ đã có ít nhất $k$ biến đúng, thì $x_i$ không được đúng nữa. Đây là clause trực tiếp ngăn tổng vượt quá $k$.

#### 5. Miền biến phụ

Chỉ tạo:

$$
s_{i,j}, \quad 1 \le j \le \min(i,k)
$$

Cách này tránh các trạng thái bất khả như $s_{1,2}$, $s_{2,3}$, ... vì với $i$ biến thì không thể có ít nhất $j$ biến đúng nếu $j>i$.

#### 6. Độ phức tạp

Số biến phụ và số clause đều là:

$$
O(nk)
$$

In [2]:
def AMK(clauses, variables, pool, k):
    """Encode At Most K using a sequential counter.

    Args:
        clauses: list[list[int]], mutated in-place.
        variables: CNF variable ids, e.g. [1, 2, 3]. Indices are 0-based.
        pool: VarPool, used to allocate fresh auxiliary variables.
        k: allowed upper bound.

    Returns:
        dict[(int, int), int]: mapping (i, j) -> auxiliary variable s_{i,j}.
        Here s[(i, j)] means variables[0..i] contain at least j + 1 true variables.
    """
    n = len(variables)

    if k < 0:
        clauses.append([])  # UNSAT: cannot choose at most a negative number of variables.
        return {}
    
    if k >= n:
        return {}

    if k == 0:
        for x_i in variables:
            clauses.append([-x_i])
        return {}

    
    s = {}


    # Create states only up to n - 2 because the last variable only needs overflow checks.
    for i in range(n - 1):
        for j in range(min(i + 1, k)):
            s[(i, j)] = pool.new_var()

    x_0 = variables[0]
    clauses.append([-x_0, s[(0, 0)]])

    for i in range(1, n - 1):
        x_i = variables[i]

        # s[(i, 0)] is true if x_i is true or the previous prefix already had one true.
        clauses.append([-x_i, s[(i, 0)]])
        clauses.append([-s[(i - 1, 0)], s[(i, 0)]])

        # Inheritance: if the previous prefix already reached j + 1 true values,
        # the current prefix also reaches j + 1 true values.
        for j in range(1, min(i - 1, k - 1) + 1):
            clauses.append([-s[(i - 1, j)], s[(i, j)]])

        # Increment: if x_i is true and the previous prefix reached j true values,
        # the current prefix reaches j + 1 true values.
        for j in range(1, min(i, k - 1) + 1):
            clauses.append([-s[(i - 1, j - 1)], -x_i, s[(i, j)]])

    # Overflow check: if the previous prefix already has k true variables, x_i must be false.
    for i in range(1, n):
        if (i - 1, k - 1) in s:
            clauses.append([-variables[i], -s[(i - 1, k - 1)]])

    return s


## 4. Giải bài toán N quân hậu bằng Sequential Counter

Gọi $q_{r,c}$ là biến Boolean: đặt quân hậu ở hàng $r$, cột $c$.

Các ràng buộc chính:
- Mỗi hàng có đúng 1 quân hậu: dùng `EXK(row, 1)`.
- Mỗi cột có đúng 1 quân hậu: dùng `EXK(col, 1)`.
- Mỗi đường chéo có nhiều nhất 1 quân hậu: dùng `AMK(diagonal, 1)`.


In [ ]:
def EXK(clauses, variables, pool, k):
    """Encode Exactly K using a sequential counter.

    Args:
        clauses: list[list[int]], mutated in-place.
        variables: CNF variable ids, e.g. [1, 2, 3]. Indices are 0-based.
        pool: VarPool, used to allocate fresh auxiliary variables.
        k: required exact number.
    """

    return {
        "at_least": ALK(clauses, variables, pool, k),
        "at_most": AMK(clauses, variables, pool, k),
    }
    

In [ ]:
from pysat.solvers import Glucose4


def n_queens_var(row, col, n):
    return row * n + col + 1


def build_n_queens_cnf(n):
    clauses = []

    pool = VarPool(n * n)

    # Each row has exactly one queen.
    for row in range(n):
        row_vars = [n_queens_var(row, col, n) for col in range(n)]
        EXK(clauses, row_vars, pool, 1)

    # Each column has exactly one queen.
    for col in range(n):
        col_vars = [n_queens_var(row, col, n) for row in range(n)]
        EXK(clauses, col_vars, pool, 1)

    # Main diagonals: row - col is constant.
    for d in range(-(n - 1), n):
        diagonal = [n_queens_var(row, row - d, n) for row in range(n) if 0 <= row - d < n]
        if len(diagonal) > 1:
            AMK(clauses, diagonal, pool, 1)

    # Anti-diagonals: row + col is constant.
    for s in range(2 * n - 1):
        diagonal = [n_queens_var(row, s - row, n) for row in range(n) if 0 <= s - row < n]
        if len(diagonal) > 1:
            AMK(clauses, diagonal, pool, 1)
    return clauses


def solve_n_queens(n):
    clauses = build_n_queens_cnf(n)

    with Glucose4(bootstrap_with=clauses) as solver:
        if not solver.solve():
            return None

        model = set(solver.get_model())

    board = []
    for row in range(n):
        board.append([
            "Q" if n_queens_var(row, col, n) in model else "."
            for col in range(n)
        ])

    return board


n = 8
solution = solve_n_queens(n)

if solution is None:
    print("UNSAT")
else:
    print("\n".join(" ".join(row) for row in solution))
